# VerticaPy MCP Server Testing Notebook

This notebook provides comprehensive testing of the VerticaPy MCP server tools.
It simulates how Claude uses the MCP server by importing and testing each tool directly.

## Setup

First, let's import all necessary modules and set up the testing environment.

In [1]:
import sys
import os
import json
import pprint
from typing import Dict, Any

# Add the MCP directory to Python path
mcp_dir = os.path.abspath('../')
if mcp_dir not in sys.path:
    sys.path.insert(0, mcp_dir)

print(f"Added to Python path: {mcp_dir}")
print(f"Current working directory: {os.getcwd()}")

Added to Python path: c:\Users\ughumman\OneDrive - OpenText\Documents\GitHub\VerticaPy\verticapy\mcp
Current working directory: c:\Users\ughumman\OneDrive - OpenText\Documents\GitHub\VerticaPy\verticapy\mcp\tests


In [2]:
# Import the MCP server functions
try:
    from server2 import (
        # Connection tools
        connect_to_vertica, disconnect_from_vertica, get_connection_status,
        # Data exploration tools
        list_tables, list_all_schemas, describe_table, sample_data,
        # Statistical analysis tools
        column_stats, table_stats,
        # Data transformation tools
        transform_data, list_cached_vdfs, clear_vdf_cache,
        # Modeling tools
        train_model, predict, list_models,
        # Query profiling tools (if available)
        create_query_profiler, get_query_plan, get_profiling_table,
        get_query_performance_summary, list_query_profilers, clear_query_profilers
    )
    print("✅ Successfully imported all MCP server functions")
except ImportError as e:
    print(f"❌ Error importing MCP server functions: {e}")
    raise

✅ Successfully imported all MCP server functions


In [3]:
# Helper functions for testing
def print_result(title: str, result: Dict[Any, Any], show_full: bool = False):
    """Pretty print test results like Claude would see them"""
    print(f"\n{'='*60}")
    print(f"🧪 {title}")
    print(f"{'='*60}")
    
    if show_full:
        pprint.pprint(result, width=80, depth=4)
    else:
        # Show key information like Claude would see
        if isinstance(result, dict):
            success = result.get('success', False)
            status = "✅ SUCCESS" if success else "❌ FAILED"
            print(f"Status: {status}")
            
            if 'error' in result:
                print(f"Error: {result['error']}")
            
            # Show truncated JSON like Claude sees
            json_str = json.dumps(result, indent=2, default=str)
            if len(json_str) > 1000:
                json_str = json_str[:1000] + "\n...\n[Output truncated]"
            print(f"\nJSON Response:\n{json_str}")
        else:
            print(f"Result: {result}")

def simulate_claude_call(tool_name: str, **kwargs):
    """Simulate how Claude calls MCP tools with JSON-RPC style logging"""
    print(f"\n📞 Simulating Claude call to tool: {tool_name}")
    print(f"Arguments: {json.dumps(kwargs, indent=2, default=str)}")
    
    # Find and call the tool function
    if tool_name in globals():
        tool_func = globals()[tool_name]
        try:
            result = tool_func(**kwargs)
            print(f"\n📤 Response from {tool_name}:")
            return result
        except Exception as e:
            error_result = {"success": False, "error": str(e)}
            print(f"\n💥 Exception in {tool_name}: {e}")
            return error_result
    else:
        print(f"❌ Tool {tool_name} not found")
        return {"success": False, "error": f"Tool {tool_name} not found"}

## 1. Connection Management Tests

First, let's test the connection management tools.

In [4]:
# Test connection status (before connecting)
result = simulate_claude_call("get_connection_status")
print_result("Get Connection Status (Initial)", result)


📞 Simulating Claude call to tool: get_connection_status
Arguments: {}

📤 Response from get_connection_status:

🧪 Get Connection Status (Initial)
Status: ❌ FAILED

JSON Response:
{
  "is_connected": false,
  "connection_name": "VerticaDSN",
  "host": "127.0.0.1",
  "port": 5433,
  "database": "demo",
  "user": "dbadmin"
}


In [5]:
# Test connection to Vertica
result = simulate_claude_call("connect_to_vertica")
print_result("Connect to Vertica", result)

# Store connection success for later tests
connection_successful = result.get('success', False)
print(f"\n🔌 Connection successful: {connection_successful}")


📞 Simulating Claude call to tool: connect_to_vertica
Arguments: {}

📤 Response from connect_to_vertica:

🧪 Connect to Vertica
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "message": "Successfully connected to Vertica database",
  "connection_info": {
    "is_connected": true,
    "connection_name": "VerticaDSN",
    "host": "127.0.0.1",
    "port": 5433,
    "database": "demo",
    "user": "dbadmin"
  },
  "verticapy_version": "1.1.1",
  "vertica_db_version": [
    23,
    4,
    0,
    0
  ]
}

🔌 Connection successful: True


c:\Users\ughumman\AppData\Local\anaconda3\envs\verticapy_dev\Lib\site-packages\vertica_python\vertica\connection.py:629: UserWarning:

TLS is not configured on the server. Proceeding with an unencrypted channel.
HINT: Set connection option 'tlsmode' to 'disable' to explicitly create a non-TLS connection.



## 2. Data Exploration Tests

Test data exploration tools like Claude would use them.

In [6]:
# Test listing schemas
if connection_successful:
    result = simulate_claude_call("list_all_schemas")
    print_result("List All Schemas", result)
    
    # Store first schema for later use
    schemas = result.get('schemas', [])
    test_schema = 'public' #schemas[0] if schemas else 'public'
    print(f"\n📋 Using schema '{test_schema}' for subsequent tests")
else:
    print("❌ Skipping schema tests - connection failed")
    test_schema = 'public'


📞 Simulating Claude call to tool: list_all_schemas
Arguments: {}

📤 Response from list_all_schemas:

🧪 List All Schemas
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "schemas": [
    "new_test",
    "public",
    "rf_test_schema",
    "test",
    "test_VYoA",
    "test_mLqw",
    "test_rf_schema",
    "test_simple",
    "test_standalone",
    "testy",
    "v_catalog",
    "v_func",
    "v_internal",
    "v_monitor",
    "v_txtindex",
    "verticapy_test_create_schema"
  ],
  "count": 16
}

📋 Using schema 'public' for subsequent tests


In [7]:
# Test listing tables
if connection_successful:
    result = simulate_claude_call("list_tables", schema=test_schema)
    print_result("List Tables", result)
    
    # Store first table for later use
    tables = result.get('tables', [])
    if tables:
        test_table = f"{test_schema}.{tables[0]}"
        print(f"\n📊 Using table '{test_table}' for subsequent tests")
    else:
        test_table = None
        print("⚠️ No tables found for testing")
else:
    print("❌ Skipping table listing - connection failed")
    test_table = None


📞 Simulating Claude call to tool: list_tables
Arguments: {
  "schema": "public"
}

📤 Response from list_tables:

🧪 List Tables
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "schema": "public",
  "tables": [
    "amazon",
    "commodities",
    "drop_data",
    "iris",
    "names_test",
    "qprof_configuration_parameters_0feeddfc98b511f08dd614755b357cb1",
    "qprof_configuration_parameters_14dcf9a298af11f098c614755b357cb1",
    "qprof_configuration_parameters_21c8258398b511f095d314755b357cb1",
    "qprof_dc_explain_plans_0feeddfc98b511f08dd614755b357cb1",
    "qprof_dc_explain_plans_14dcf9a298af11f098c614755b357cb1",
    "qprof_dc_explain_plans_21c8258398b511f095d314755b357cb1",
    "qprof_dc_lock_attempts_0feeddfc98b511f08dd614755b357cb1",
    "qprof_dc_lock_attempts_14dcf9a298af11f098c614755b357cb1",
    "qprof_dc_lock_attempts_21c8258398b511f095d314755b357cb1",
    "qprof_dc_plan_activities_0feeddfc98b511f08dd614755b357cb1",
    "qprof_dc_plan_activities_14dcf9a298af11f

In [8]:
test_table = 'public.winequality'  # Set a default test table for consistency

In [9]:
# Test describing a table (like winequality in Claude logs)
if connection_successful and test_table:
    result = simulate_claude_call("describe_table", table=test_table)
    print_result("Describe Table", result)
    
    # Store column info for later statistical tests
    columns_info = result.get('columns', [])
    if columns_info:
        print(f"📋 Debug: columns_info type: {type(columns_info)}")
        print(f"📋 Debug: columns_info content: {columns_info}")
        
        try:
            # Handle different column info formats
            if isinstance(columns_info, list) and len(columns_info) > 0:
                # Check what the first element looks like
                first_element = columns_info[0]
                if isinstance(first_element, dict) and 'name' in first_element:
                    # If columns are dictionaries with 'name' key
                    test_columns = [col['name'].strip('"') for col in columns_info]
                elif isinstance(first_element, str):
                    # If columns are just strings (column names)
                    test_columns = [col.strip('"') for col in columns_info]
                else:
                    # Fallback - try to convert to string
                    test_columns = [str(col).strip('"') for col in columns_info]
            elif isinstance(columns_info, dict):
                # If columns_info is a dictionary, try to extract column names
                if 'index' in columns_info and isinstance(columns_info['index'], list):
                    # Column names are in the 'index' key
                    test_columns = [str(col).strip('"') for col in columns_info['index']]
                elif 'columns' in columns_info:
                    test_columns = [str(col).strip('"') for col in columns_info['columns']]
                else:
                    # Use dictionary keys as column names (fallback)
                    test_columns = [str(key).strip('"') for key in columns_info.keys()]
            else:
                # Convert whatever it is to a list of strings
                test_columns = [str(item).strip('"') for item in columns_info]
                
            print(f"\n📈 Available columns for stats: {test_columns[:5]}...")  # Show first 5
        except Exception as e:
            print(f"⚠️ Error parsing columns: {e}")
            test_columns = []
    else:
        test_columns = []
else:
    print("❌ Skipping table description - no test table available")
    test_columns = []


📞 Simulating Claude call to tool: describe_table
Arguments: {
  "table": "public.winequality"
}

📤 Response from describe_table:

🧪 Describe Table
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "table": "public.winequality",
  "row_count": 6497,
  "column_count": 14,
  "columns": {
    "index": [
      "fixed_acidity",
      "volatile_acidity",
      "citric_acid",
      "residual_sugar",
      "chlorides",
      "free_sulfur_dioxide",
      "total_sulfur_dioxide",
      "density",
      "pH",
      "sulphates",
      "alcohol",
      "quality",
      "good",
      "color"
    ],
    "dtype": [
      "numeric(6,3)",
      "numeric(7,4)",
      "numeric(6,3)",
      "numeric(7,3)",
      "float",
      "numeric(7,2)",
      "numeric(7,2)",
      "float",
      "numeric(6,3)",
      "numeric(6,3)",
      "float",
      "int",
      "int",
      "varchar(20)"
    ]
  },
  "stats": {
    "index": [
      "fixed_acidity",
      "volatile_acidity",
      "citric_acid",
      "resi

In [10]:
test_columns

['fixed_acidity',
 'volatile_acidity',
 'citric_acid',
 'residual_sugar',
 'chlorides',
 'free_sulfur_dioxide',
 'total_sulfur_dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'quality',
 'good',
 'color']

In [11]:
# Test sampling data
if connection_successful and test_table:
    result = simulate_claude_call("sample_data", table=test_table, n=5)
    print_result("Sample Data", result)
else:
    print("❌ Skipping data sampling - no test table available")


📞 Simulating Claude call to tool: sample_data
Arguments: {
  "table": "public.winequality",
  "n": 5
}

📤 Response from sample_data:

🧪 Sample Data
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "table": "public.winequality",
  "columns": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    "quality",
    "good",
    "color"
  ],
  "data": [
    {
      "fixed_acidity": 3.8,
      "volatile_acidity": 0.31,
      "citric_acid": 0.02,
      "residual_sugar": 11.1,
      "chlorides": 0.036,
      "free_sulfur_dioxide": 20.0,
      "total_sulfur_dioxide": 114.0,
      "density": 0.99248,
      "pH": 3.75,
      "sulphates": 0.44,
      "alcohol": 12.4,
      "quality": 6,
      "good": 0,
      "color": "white"
    },
    {
      "fixed_acidity": 3.9,
      "volatile_acidity": 0.225,
      "citric_acid": 0.4,
    

## 3. Statistical Analysis Tests

Test the statistical analysis tools exactly like Claude used them in the logs.

In [12]:
# Test column statistics - topk like in Claude logs
if connection_successful and test_table and test_columns:
    # Try to find a categorical column for topk test
    test_column = test_columns[0]  # Use first column
    
    # Simulate Claude's call: column_stats with topk and extra_kwargs
    result = simulate_claude_call(
        "column_stats",
        table=test_table,
        column=test_column,
        metric="topk",
        extra_kwargs='{"k": 5}'
    )
    print_result(f"Column Stats - TopK for {test_column}", result)
else:
    print("❌ Skipping column stats - no test data available")


📞 Simulating Claude call to tool: column_stats
Arguments: {
  "table": "public.winequality",
  "column": "fixed_acidity",
  "metric": "topk",
  "extra_kwargs": "{\"k\": 5}"
}

📤 Response from column_stats:

🧪 Column Stats - TopK for fixed_acidity
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "table": "public.winequality",
  "column": "fixed_acidity",
  "metric": "topk",
  "result": {
    "index": [
      6.8,
      6.6,
      6.4,
      7,
      6.9
    ],
    "count": [
      354,
      327,
      305,
      282,
      279
    ],
    "percent": [
      5.449,
      5.033,
      4.694,
      4.34,
      4.294
    ]
  },
  "method": "vDataColumn"
}


In [13]:
# Test different metrics like Claude would
if connection_successful and test_table and test_columns:
    # Test multiple statistical metrics
    metrics_to_test = ['describe', 'count', 'nunique', 'mean']
    
    for metric in metrics_to_test:
        if len(test_columns) > 1:
            test_column = test_columns[1]  # Use second column
        else:
            test_column = test_columns[0]
            
        result = simulate_claude_call(
            "column_stats",
            table=test_table,
            column=test_column,
            metric=metric
        )
        print_result(f"Column Stats - {metric.upper()} for {test_column}", result, show_full=False)
        
        # Brief pause between calls
        import time
        time.sleep(0.5)
else:
    print("❌ Skipping multiple column stats - no test data available")


📞 Simulating Claude call to tool: column_stats
Arguments: {
  "table": "public.winequality",
  "column": "volatile_acidity",
  "metric": "describe"
}

📤 Response from column_stats:

🧪 Column Stats - DESCRIBE for volatile_acidity
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "table": "public.winequality",
  "column": "volatile_acidity",
  "metric": "describe",
  "result": {
    "index": [
      "name",
      "dtype",
      "unique",
      "count",
      "mean",
      "std",
      "min",
      "approx_25%",
      "approx_50%",
      "approx_75%",
      "max"
    ],
    "value": [
      "\"volatile_acidity\"",
      "numeric(7,4)",
      186.0,
      6497,
      0.339665999692165,
      0.164636474084679,
      0.08,
      0.23,
      0.29,
      0.4,
      1.58
    ]
  },
  "method": "vDataColumn"
}

📞 Simulating Claude call to tool: column_stats
Arguments: {
  "table": "public.winequality",
  "column": "volatile_acidity",
  "metric": "count"
}

📤 Response from column_stats:


In [14]:
# Test table-level statistics
if connection_successful and test_table:
    result = simulate_claude_call(
        "table_stats",
        table=test_table,
        metric="describe"
    )
    print_result("Table Stats - Describe", result)
else:
    print("❌ Skipping table stats - no test table available")


📞 Simulating Claude call to tool: table_stats
Arguments: {
  "table": "public.winequality",
  "metric": "describe"
}

📤 Response from table_stats:

🧪 Table Stats - Describe
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "table": "public.winequality",
  "metric": "describe",
  "columns_analyzed": "auto_selected",
  "result": {
    "index": [
      "\"fixed_acidity\"",
      "\"volatile_acidity\"",
      "\"citric_acid\"",
      "\"residual_sugar\"",
      "\"chlorides\"",
      "\"free_sulfur_dioxide\"",
      "\"total_sulfur_dioxide\"",
      "\"density\"",
      "\"pH\"",
      "\"sulphates\"",
      "\"alcohol\"",
      "\"quality\"",
      "\"good\""
    ],
    "count": [
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497,
      6497
    ],
    "mean": [
      7.21530706479914,
      0.339665999692165,
      0.318633215330153,
      5.44323533938742,
      0.0560338617823611,
     

## 4. Machine Learning Tests

Test model training exactly like Claude did in the logs.

In [16]:
# Test model training like Claude's wine quality example
import verticapy as vp
vp.drop("test_model_notebook")
if connection_successful and test_table and len(test_columns) >= 3:
    # Select features and target (assuming last column is target)
    features =     [
        "fixed_acidity",
        "volatile_acidity",
        "citric_acid",
        "residual_sugar",
        "chlorides",
        "density",
    ] #test_columns[:-1][:6]  # Use first 6 columns as features
    target = "quality" #test_columns[-1]  # Last column as target
    
    print(f"\n🤖 Training model with:")
    print(f"   Features: {features}")
    print(f"   Target: {target}")
    
    # Simulate Claude's train_model call
    result = simulate_claude_call(
        "train_model",
        table=test_table,
        model_type="linear_regression",
        model_name="test_model_notebook",
        target=target,
        features=features,
        extra_kwargs="{}"
    )
    print_result("Train Linear Regression Model", result)
    
    # Store model info for prediction test
    model_trained = result.get('success', False)
    model_name = "test_model_notebook" if model_trained else None
    
else:
    print("❌ Skipping model training - insufficient columns or no connection")
    model_trained = False
    model_name = None


🤖 Training model with:
   Features: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'density']
   Target: quality

📞 Simulating Claude call to tool: train_model
Arguments: {
  "table": "public.winequality",
  "model_type": "linear_regression",
  "model_name": "test_model_notebook",
  "target": "quality",
  "features": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "density"
  ],
  "extra_kwargs": "{}"
}

📤 Response from train_model:

🧪 Train Linear Regression Model
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "model_info": {
    "model_name": "test_model_notebook",
    "model_type": "linear_regression",
    "features": [
      "\"fixed_acidity\"",
      "\"volatile_acidity\"",
      "\"citric_acid\"",
      "\"residual_sugar\"",
      "\"chlorides\"",
      "\"density\""
    ],
    "n_features": 6,
    "training_table": "table 'public.winequality'",
    "model_stored": true,
 

In [17]:
# Test model listing
if connection_successful:
    result = simulate_claude_call("list_models", limit=10)
    print_result("List Models", result)
else:
    print("❌ Skipping model listing - no connection")


📞 Simulating Claude call to tool: list_models
Arguments: {
  "limit": 10
}

📤 Response from list_models:

🧪 List Models
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "models": [
    {
      "schema_name": "public",
      "model_name": "test_model_notebook",
      "full_name": "public.test_model_notebook",
      "model_type": "LINEAR_REGRESSION",
      "category": "VERTICA_MODELS",
      "owner": "dbadmin",
      "created_at": "2025-10-02 20:46:22.038627+00:00",
      "summary_available": true,
      "has_predictors": true
    },
    {
      "schema_name": "new_test",
      "model_name": "model1",
      "full_name": "new_test.model1",
      "model_type": "LOGISTIC_REGRESSION",
      "category": "VERTICA_MODELS",
      "owner": "dbadmin",
      "created_at": "2025-10-01 15:34:40.080171+00:00",
      "summary_available": true,
      "has_predictors": true
    },
    {
      "schema_name": "public",
      "model_name": "model_titanic_test_rfc",
      "full_name": "public.model_

In [18]:
# Test prediction if model was trained
if model_trained and model_name and test_table:
    result = simulate_claude_call(
        "predict",
        table=test_table,
        model_name=model_name,
        output_name="predicted_values"
    )
    print_result("Make Predictions", result)
else:
    print("❌ Skipping prediction - no trained model available")


📞 Simulating Claude call to tool: predict
Arguments: {
  "table": "public.winequality",
  "model_name": "test_model_notebook",
  "output_name": "predicted_values"
}

📤 Response from predict:

🧪 Make Predictions
Status: ✅ SUCCESS

JSON Response:
{
  "success": true,
  "model_name": "test_model_notebook",
  "prediction_table": "table 'public.winequality'",
  "prediction_column": "predicted_values",
  "prediction_type": "prediction",
  "total_predictions": 6497,
  "result_columns": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    "quality",
    "good",
    "color",
    "predicted_values"
  ],
  "sample_predictions": [
    {
      "fixed_acidity": 3.8,
      "volatile_acidity": 0.31,
      "citric_acid": 0.02,
      "residual_sugar": 11.1,
      "chlorides": 0.036,
      "free_sulfur_dioxide": 20.0,
      "total_sulfur_

## 5. Data Transformation Tests

Test data transformation capabilities.

In [ ]:
# Test data transformation - groupby
if connection_successful and test_table and test_columns:
    # Create a simple groupby transformation
    group_column = test_columns[0]  # Use first column for grouping
    
    result = simulate_claude_call(
        "transform_data",
        table=test_table,
        operation="groupby",
        vdf_id="test_groupby",
        show_preview=True,
        extra_kwargs=json.dumps({
            "columns": [group_column],
            "expr": ["COUNT(*) AS record_count"]
        })
    )
    print_result(f"Data Transformation - GroupBy {group_column}", result)
else:
    print("❌ Skipping data transformation - no test data available")

In [ ]:
# Test listing cached vDataFrames
result = simulate_claude_call("list_cached_vdfs")
print_result("List Cached vDataFrames", result)

## 6. Query Profiling Tests

Test query profiling tools if available.

In [ ]:
# Test query profiler creation
if connection_successful and test_table:
    simple_query = f"SELECT COUNT(*) FROM {test_table}"
    
    result = simulate_claude_call(
        "create_query_profiler",
        source_type="query",
        query=simple_query,
        target_schema=test_schema
    )
    print_result("Create Query Profiler", result)
    
    profiler_created = result.get('success', False)
    profiler_id = result.get('profiler_info', {}).get('profiler_id') if profiler_created else None
else:
    print("❌ Skipping query profiler - no connection or test table")
    profiler_created = False
    profiler_id = None

## 7. Test Summary and Cleanup

Provide a summary of all tests and clean up resources.

In [ ]:
# Test summary
print("\n" + "="*80)
print("🏁 TEST SUMMARY")
print("="*80)

test_results = {
    "Connection": connection_successful,
    "Data Exploration": connection_successful and test_table is not None,
    "Statistical Analysis": connection_successful and len(test_columns) > 0,
    "Machine Learning": model_trained,
    "Query Profiling": profiler_created
}

for test_category, success in test_results.items():
    status = "✅ PASSED" if success else "❌ FAILED/SKIPPED"
    print(f"{test_category:20}: {status}")

print("\n" + "="*80)
print("📊 Overall Test Status:")
passed_tests = sum(test_results.values())
total_tests = len(test_results)
print(f"   {passed_tests}/{total_tests} test categories passed")

if passed_tests == total_tests:
    print("   🎉 ALL TESTS PASSED!")
elif passed_tests > 0:
    print("   ⚠️  PARTIAL SUCCESS - Some tests failed or were skipped")
else:
    print("   💥 ALL TESTS FAILED - Check connection and setup")

In [ ]:
# Cleanup - Clear caches and disconnect
print("\n🧹 Cleaning up...")

# Clear vDataFrame cache
clear_result = simulate_claude_call("clear_vdf_cache")
print(f"Clear vDF Cache: {'✅' if clear_result.get('success') else '❌'}")

# Clear query profilers
if profiler_created:
    clear_profilers_result = simulate_claude_call("clear_query_profilers")
    print(f"Clear Query Profilers: {'✅' if clear_profilers_result.get('success') else '❌'}")

# Disconnect
if connection_successful:
    disconnect_result = simulate_claude_call("disconnect_from_vertica")
    print(f"Disconnect: {'✅' if disconnect_result.get('success') else '❌'}")

print("\n✨ Testing complete!")